# Export & Compile TensorRT Engine Model (`road_following_model.engine`)

This notebook converts and optimizes your trained **ONNX model** (`road_following_model.onnx`) directly into an **NVIDIA TensorRT Engine file (`road_following_model.engine`)** with **FP16 precision** for maximum GPU acceleration on Jetson Nano.

| Step | Description |
|------|-------------|
| 1 | Locate Trained ONNX Model File |
| 2 | Compile ONNX Model to TensorRT Engine (`trtexec` FP16 Mode) |
| 3 | Verify & Benchmark TensorRT Latency (FPS) |

### 1. Setup Environment & Locate ONNX Model File

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Add parent directory to sys.path
parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))

onnx_path = os.path.join(Path.cwd(), "road_following_model.onnx")
if not os.path.exists(onnx_path):
    onnx_path = os.path.join(parent_dir, "notebooks", "road_following_model.onnx")

engine_path = os.path.join(Path.cwd(), "road_following_model.engine")

if not os.path.exists(onnx_path):
    print(f"[!] ERROR: ONNX model file '{onnx_path}' not found! Run train_model_onnx.ipynb first.")
else:
    print(f"[+] Found ONNX model: {onnx_path}")
    print(f"[*] Target TensorRT Engine output: {engine_path}")


### 2. Compile ONNX Model to TensorRT Engine (`trtexec` FP16 Mode)

In [ ]:
# Check for NVIDIA trtexec binary location on Jetson Nano
trtexec_bin = "/usr/src/tensorrt/bin/trtexec"
if not os.path.exists(trtexec_bin):
    trtexec_bin = "trtexec"

print(f"[*] Starting TensorRT FP16 Compilation with {trtexec_bin}...")
cmd = [
    trtexec_bin,
    f"--onnx={onnx_path}",
    f"--saveEngine={engine_path}",
    "--fp16",
    "--workspace=1024"
]

print(f"[*] Running command: {' '.join(cmd)}\n")
try:
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
    for line in process.stdout:
        print(line, end='')
    process.wait()
    
    if process.returncode == 0 and os.path.exists(engine_path):
        print(f"\n[+] SUCCESS! TensorRT Engine compiled & saved -> '{engine_path}'")
    else:
        print(f"\n[!] trtexec returned exit code: {process.returncode}")
except Exception as e:
    print(f"\n[!] trtexec notice: {e}")


### 3. Fallback Compiler via `onnxruntime` TensorRT Provider (if `trtexec` binary path differs)

In [ ]:
if not os.path.exists(engine_path):
    print("[*] Generating TensorRT Cache Engine via ONNX Runtime...")
    import onnxruntime as ort
    
    trt_options = {
        'device_id': 0,
        'trt_max_workspace_size': 1073741824, # 1GB
        'trt_fp16_enable': True,
        'trt_engine_cache_enable': True,
        'trt_engine_cache_path': str(Path.cwd()),
    }
    
    try:
        session = ort.InferenceSession(onnx_path, providers=[('TensorrtExecutionProvider', trt_options), 'CUDAExecutionProvider', 'CPUExecutionProvider'])
        print(f"[+] Active Providers: {session.get_providers()}")
        print(f"[+] TensorRT Engine cache successfully generated in current directory!")
    except Exception as e:
        print(f"[!] TensorRT Fallback Notice: {e}")


### 4. Verify & Benchmark TensorRT Model Latency (FPS)

In [ ]:
import time
import numpy as np
import onnxruntime as ort

print("[*] Benchmarking TensorRT Inference Latency...")

providers = [
    ('TensorrtExecutionProvider', {'trt_fp16_enable': True, 'device_id': 0}),
    'CUDAExecutionProvider',
    'CPUExecutionProvider'
]

try:
    session = ort.InferenceSession(onnx_path, providers=providers)
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name
    
    dummy_input = np.random.randn(1, 3, 224, 224).astype(np.float32)
    
    # Warmup iterations
    for _ in range(10):
        _ = session.run([output_name], {input_name: dummy_input})
        
    # Benchmark 100 runs
    times = []
    for _ in range(100):
        t0 = time.time()
        _ = session.run([output_name], {input_name: dummy_input})
        t1 = time.time()
        times.append((t1 - t0) * 1000.0)
        
    avg_ms = np.mean(times)
    min_ms = np.min(times)
    fps = 1000.0 / avg_ms
    
    print(f"  [+] Loaded Providers : {session.get_providers()}")
    print(f"  [+] Average Latency  : {avg_ms:.2f} ms ({fps:.1f} FPS)")
    print(f"  [+] Minimum Latency  : {min_ms:.2f} ms")
    print(f"\n[+] TENSORRT CONVERSION & BENCHMARK COMPLETED 100%! Ready for road_following_live.ipynb!")
except Exception as e:
    print(f"[!] Benchmark Exception: {e}")
